Cell 0 — Parameters (Papermill tag: parameters)
This cell is tagged parameters in Jupyter so Papermill can inject overrides here at runtime. That tag is set in the notebook metadata, not in the code itself — in JupyterLab: right-click the cell → Add Tag → type parameters.

In [ ]:
# Cell 0 — tagged: parameters
# Papermill injects overrides here. Defaults match scad/params.json.
# To run a sweep: papermill 01_geometry_openscad.ipynb out.ipynb -p part_name "sweep_01"

import sys
sys.path.insert(0, "/workspace")  # ensure src/ is importable inside Docker

PARAMS_JSON   = "scad/params.json"
SCAD_FILE     = "scad/base_part.scad"
TIMEOUT_S     = 120
PART_NAME_OVERRIDE = None  # set by Papermill e.g. "bracket_narrow"

Cell 1 — Load and validate params

In [ ]:
# Cell 1 — Load params and validate schema
from src.geometry.param_schema import PipelineParams

params = PipelineParams.from_json(PARAMS_JSON)

# Apply Papermill override if provided
if PART_NAME_OVERRIDE is not None:
    params.part_name = PART_NAME_OVERRIDE

# Validate — raises AssertionError with a clear message if anything is wrong
params.validate()

print(f"part_name:    {params.part_name}")
print(f"geometry:     {params.geometry}")
print(f"mesh_hints:   {params.mesh_hints}")
print(f"load_hints:   {params.load_hints}")
print(f"export dir:   {params.export.stl_output_dir}")

Cell 2 — Show OpenSCAD defines
Inspectable checkpoint — you can verify exactly what flags will be passed to OpenSCAD before running it.

In [ ]:
# Cell 2 — Inspect defines before running OpenSCAD
defines = params.to_openscad_defines()
print("OpenSCAD -D defines:")
for k, v in defines.items():
    print(f"  {k:<30} = {v}")

Cell 3 — Run OpenSCAD

In [ ]:
# Cell 3 — Compile geometry
from src.geometry.openscad_runner import run_openscad

result = run_openscad(params, scad_file=SCAD_FILE, timeout_s=TIMEOUT_S)

print(f"Success:    {result.success}")
print(f"Duration:   {result.duration_s}s")
print(f"STL path:   {result.stl_path}")
print(f"STL size:   {result.stl_size_kb} KB")
if result.stderr:
    print(f"\nStderr:\n{result.stderr}")

# Hard fail here — don't let a bad STL silently propagate to Stage 2
result.raise_if_failed()

Cell 4 — Quick mesh preview
Renders headlessly via PyVista and saves a PNG to outputs/reports/. This gives you a visual sanity check without needing a display.

In [ ]:
# Cell 4 — Headless STL preview → outputs/reports/
import pyvista as pv
from pathlib import Path

pv.OFF_SCREEN = True

mesh = pv.read(str(result.stl_path))
print(f"Vertices:  {mesh.n_points}")
print(f"Faces:     {mesh.n_cells}")
print(f"Bounds:    {[round(b,2) for b in mesh.bounds]}")

report_dir = Path("outputs/reports")
report_dir.mkdir(parents=True, exist_ok=True)
png_path = report_dir / f"{params.part_name}_geometry.png"

pl = pv.Plotter()
pl.add_mesh(mesh, color="lightsteelblue", show_edges=True)
pl.add_axes()
pl.screenshot(str(png_path))
pl.close()

print(f"\nPreview saved: {png_path}")

Cell 5 — Export stage output for handoff
Writes a small JSON sidecar next to the STL so Stage 2 (02_mesh_gmsh.ipynb) knows where to find the geometry and what parameters were used, without having to re-read params.json.

In [ ]:
# Cell 5 — Write stage handoff sidecar
# 02_mesh_gmsh.ipynb reads this to find the STL and mesh hints
import json
from pathlib import Path
from dataclasses import asdict

handoff = {
    "stage":        "01_geometry",
    "part_name":    params.part_name,
    "stl_path":     str(result.stl_path),
    "stl_size_kb":  result.stl_size_kb,
    "duration_s":   result.duration_s,
    "mesh_hints":   asdict(params.mesh_hints),
    "load_hints":   asdict(params.load_hints),
}

handoff_path = Path(params.export.stl_output_dir) / f"{params.part_name}_stage01.json"
handoff_path.write_text(json.dumps(handoff, indent=2))
print(f"Handoff written: {handoff_path}")
print(json.dumps(handoff, indent=2))

How these files connect to the rest of the pipeline
The sidecar JSON from Cell 5 is the contract between Stage 1 and Stage 2. When 02_mesh_gmsh.ipynb runs, its Cell 0 reads outputs/meshes/<part_name>_stage01.json rather than re-parsing params.json — this means even if params change mid-run, Stage 2 always meshes the geometry that was actually compiled, not whatever the current params say.
The mesh_hints.target_element_size value travels through this sidecar into src/meshing/gmsh_pipeline.py in Stage 2, where it controls the global mesh size. The load_hints travel all the way to src/fea/boundary_conditions.py in Stage 3.
Ready to move to Stage 2 — 02_mesh_gmsh.ipynb and src/meshing/?